# Initial EDA: `data/raw/pre-combine`

Notebook นี้ใช้สำหรับสำรวจไฟล์ดิบแบบเบื้องต้นก่อนนำไป combine หรือ preprocess ต่อ โดยเน้นดูโครงสร้างไฟล์ ตัวอย่างข้อมูล ค่าหาย ค่าซ้ำ และค่าสถิติพื้นฐานแบบเร็วๆ

ขอบเขตใน notebook นี้:
- อ่านไฟล์จาก `data/raw/pre-combine` แบบ recursive
- แยก section ตามแต่ละไฟล์
- ยังไม่ทำ feature engineering, cleaning, modeling หรือสรุปเชิงลึก


In [1]:
%pip install pandas

  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 33.1 MB/s  0:00:00eta 0:00:01
Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "pre-combine"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"ไม่พบโฟลเดอร์ข้อมูล: {DATA_DIR}")

print(f"Project root: {PROJECT_ROOT}")
print(f"EDA data dir: {DATA_DIR}")


Project root: /Users/pattamaponthongplew/Documents/GitHub/crop-recommendation-system-for-smart-farming
EDA data dir: /Users/pattamaponthongplew/Documents/GitHub/crop-recommendation-system-for-smart-farming/data/raw/pre-combine


## 1. ภาพรวมไฟล์ทั้งหมด

cell นี้สแกนไฟล์ข้อมูลทั้งหมดใน `data/raw/pre-combine` เพื่อดูว่ามีไฟล์อะไรบ้าง อยู่ในโฟลเดอร์ไหน และมีขนาดไฟล์ประมาณเท่าไร


In [3]:
DATA_FILES = sorted(
    [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in {".csv", ".tsv", ".txt"}]
)

file_overview = pd.DataFrame({
    "folder": [str(p.parent.relative_to(DATA_DIR)) for p in DATA_FILES],
    "file": [p.name for p in DATA_FILES],
    "extension": [p.suffix.lower() for p in DATA_FILES],
    "size_mb": [p.stat().st_size / (1024 ** 2) for p in DATA_FILES],
    "path": [str(p.relative_to(PROJECT_ROOT)) for p in DATA_FILES],
})

print(f"พบไฟล์ข้อมูลทั้งหมด {len(DATA_FILES)} ไฟล์")
display(file_overview.sort_values(["folder", "file"]).reset_index(drop=True))


พบไฟล์ข้อมูลทั้งหมด 9 ไฟล์


,folder,file,extension,size_mb,path
0,.,Crop Recommendation using Soil Properties and ...,.csv,0.8360,data/raw/pre-combine/Crop Recommendation using...
1,.,Crop and fertilizer dataset.csv,.csv,0.3594,data/raw/pre-combine/Crop and fertilizer datas...
2,.,Crop_Recommendation.csv,.csv,0.1360,data/raw/pre-combine/Crop_Recommendation.csv
3,.,Crop_recommendation-3.csv,.csv,0.1410,data/raw/pre-combine/Crop_recommendation-3.csv
4,.,crop-yield.csv,.csv,1.0384,data/raw/pre-combine/crop-yield.csv
5,.,crop_data.csv,.csv,0.0237,data/raw/pre-combine/crop_data.csv
6,.,crop_yield.csv,.csv,2.0617,data/raw/pre-combine/crop_yield.csv
7,.,data_core.csv,.csv,0.3497,data/raw/pre-combine/data_core.csv
8,.,original_dataset.csv,.csv,0.2263,data/raw/pre-combine/original_dataset.csv


## 2. Helper functions

ฟังก์ชันชุดนี้ใช้ซ้ำกับทุกไฟล์ เพื่อให้ EDA แต่ละไฟล์มีรูปแบบเดียวกัน ได้แก่ การอ่านไฟล์ สรุปโครงสร้าง ตรวจ missing/duplicate และดูสถิติพื้นฐาน


In [4]:
def read_table(path: Path) -> pd.DataFrame:
    """Read a table with a few common encoding fallbacks."""
    sep = "	" if path.suffix.lower() == ".tsv" else ","
    last_error = None
    for encoding in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            return pd.read_csv(path, sep=sep, encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def file_meta(path: Path, df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "metric": ["relative_path", "size_mb", "rows", "columns", "memory_mb", "exact_duplicate_rows"],
        "value": [
            str(path.relative_to(PROJECT_ROOT)),
            round(path.stat().st_size / (1024 ** 2), 4),
            len(df),
            len(df.columns),
            round(df.memory_usage(deep=True).sum() / (1024 ** 2), 4),
            int(df.duplicated().sum()),
        ],
    })


def column_profile(df: pd.DataFrame) -> pd.DataFrame:
    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100),
        "unique_count": df.nunique(dropna=True).values,
    })
    profile["missing_pct"] = profile["missing_pct"].round(2)
    return profile


def display_structure(path: Path, df: pd.DataFrame) -> None:
    display(Markdown(f"### โครงสร้างไฟล์: `{path.name}`"))
    display(file_meta(path, df))
    display(Markdown("**Columns / dtypes / missing / unique**"))
    display(column_profile(df))


def display_missing_duplicates(df: pd.DataFrame) -> None:
    display(Markdown("### Missing values และ duplicate rows"))
    missing = column_profile(df).query("missing_count > 0").sort_values(
        ["missing_pct", "missing_count"], ascending=False
    )
    if missing.empty:
        print("ไม่พบ missing values")
    else:
        display(missing)

    dup_count = int(df.duplicated().sum())
    print(f"Exact duplicate rows: {dup_count}")
    if dup_count:
        display(df[df.duplicated(keep=False)].head(10))


def display_samples_and_stats(df: pd.DataFrame) -> None:
    display(Markdown("### ตัวอย่างข้อมูลและสถิติพื้นฐาน"))
    display(Markdown("**Head sample**"))
    display(df.head())

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    object_cols = df.select_dtypes(exclude="number").columns.tolist()

    if numeric_cols:
        display(Markdown("**Numeric summary**"))
        display(df[numeric_cols].describe().T)
    else:
        print("ไม่มี numeric columns")

    if object_cols:
        display(Markdown("**Categorical summary**"))
        display(df[object_cols].describe().T)

        focus_cols = [c for c in ["label", "crop", "state", "season", "soil_type", "soil_color"] if c in df.columns]
        for col in focus_cols[:4]:
            display(Markdown(f"**Top values: `{col}`**"))
            display(df[col].value_counts(dropna=False).head(10).to_frame("count"))
    else:
        print("ไม่มี categorical/text columns")


## File 1: `Crop Recommendation using Soil Properties and Weather Prediction.csv`

Folder: `.`  
Path: `data/raw/pre-combine/Crop Recommendation using Soil Properties and Weather Prediction.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [5]:
# File 1: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/Crop Recommendation using Soil Properties and Weather Prediction.csv'
df_01 = read_table(path)
display_structure(path, df_01)


### โครงสร้างไฟล์: `Crop Recommendation using Soil Properties and Weather Prediction.csv`

,metric,value
0,relative_path,data/raw/pre-combine/Crop Recommendation using...
1,size_mb,0.8360
2,rows,3867
3,columns,29
4,memory_mb,1.2531
5,exact_duplicate_rows,0


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,12
1,n,float64,0,0.0000,1348
2,p,float64,0,0.0000,957
3,k,float64,0,0.0000,1206
4,ph,float64,0,0.0000,281
5,soil_color,str,0,0.0000,45
6,Zinc,float64,0,0.0000,624
7,Sulfur,float64,0,0.0000,805
8,Humidity_Winter,float64,0,0.0000,15
9,Humidity_Spring,float64,0,0.0000,15


In [6]:
# File 1: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_01)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 0


In [7]:
# File 1: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_01)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,ph,soil_color,Zinc,Sulfur,Humidity_Winter,Humidity_Spring,Humidity_Summer,Humidity_Autumn,Max_Temp_Winter,Max_Temp_Spring,Max_Temp_Summer,Max_Temp_Autumn,Min_Temp_Winter,Min_Temp_Spring,Min_Temp_Summer,Min_Temp_Autumn,Precipitation_Winter,Precipitation_Spring,Precipitation_Summer,Precipitation_Autumn,Wind_Direction,soil_moisture,Cloud_Amount,Wind_Speed_Range,Surface_Pressure
0,barley,0.2300,5.4010,738.2310,5.8100,Yellowish brown,2.9760,13.8160,7.9933,10.4567,11.9633,9.6833,26.8533,28.5267,23.0600,22.2733,5.3900,9.8900,10.4167,5.6933,2.0733,5.2700,12.3033,5.2700,3.4400,0.7300,56.5700,6.2400,77.0300
1,barley,0.2300,10.4780,606.3820,5.4300,Yellowish brown,3.0770,16.4210,7.9933,10.4567,11.9633,9.6833,26.8533,28.5267,23.0600,22.2733,5.3900,9.8900,10.4167,5.6933,2.0733,5.2700,12.3033,5.2700,3.4400,0.7300,56.5700,6.2400,77.0300
2,barley,0.2300,6.8470,386.5800,5.4100,brown,6.6110,16.5570,7.9933,10.4567,11.9633,9.6833,26.8533,28.5267,23.0600,22.2733,5.3900,9.8900,10.4167,5.6933,2.0733,5.2700,12.3033,5.2700,3.4400,0.7300,56.5700,6.2400,77.0300
3,barley,0.2300,3.4180,207.0860,5.6500,red,0.4602,16.0750,7.9933,10.4567,11.9633,9.6833,26.8533,28.5267,23.0600,22.2733,5.3900,9.8900,10.4167,5.6933,2.0733,5.2700,12.3033,5.2700,3.4400,0.7300,56.5700,6.2400,77.0300
4,barley,0.2300,39.2820,317.3570,5.2700,red,2.7430,12.5580,7.9933,10.4567,11.9633,9.6833,26.8533,28.5267,23.0600,22.2733,5.3900,9.8900,10.4167,5.6933,2.0733,5.2700,12.3033,5.2700,3.4400,0.7300,56.5700,6.2400,77.0300


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,"3,867.0000",0.1792,0.0665,0.0003,0.1312,0.1799,0.2300,0.6956
p,"3,867.0000",11.3496,34.1419,0.0000,2.0000,4.0000,7.9200,782.0000
k,"3,867.0000",324.2848,202.2501,41.1340,191.0000,282.0000,405.0000,"2,119.0000"
ph,"3,867.0000",5.8573,0.6767,4.3000,5.3900,5.7800,6.2000,8.5000
Zinc,"3,867.0000",1.7741,1.4608,0.1000,1.1000,1.5000,2.0600,45.5000
Sulfur,"3,867.0000",11.3116,5.5421,0.0500,7.3050,10.7000,14.1955,118.3470
Humidity_Winter,"3,867.0000",8.3469,0.6139,7.1833,7.9333,8.3833,8.9100,9.7233
Humidity_Spring,"3,867.0000",9.0825,0.7907,7.6500,8.4033,9.3400,9.4800,10.7033
Humidity_Summer,"3,867.0000",12.2320,0.8950,10.4767,11.4300,12.1667,12.8367,13.8533
Humidity_Autumn,"3,867.0000",10.9907,0.9939,9.0700,10.1500,10.9267,11.4333,12.8000


...

,count,unique,top,freq
label,3867,12,teff,1260
soil_color,3867,45,brown,1297


**Top values: `label`**

,count
label,
teff,1260
maize,732
wheat,715
barley,503
bean,253
pea,94
sorghum,72
dagussa,71
niger_seed,64


**Top values: `soil_color`**

,count
soil_color,
brown,1297
red,1195
black,860
other,128
gray,88
reddish brown,85
dark brown,51
dark gray,35
Reddish brown,14


## File 2: `Crop and fertilizer dataset.csv`

Folder: `.`  
Path: `data/raw/pre-combine/Crop and fertilizer dataset.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [8]:
# File 2: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/Crop and fertilizer dataset.csv'
df_02 = read_table(path)
display_structure(path, df_02)


### โครงสร้างไฟล์: `Crop and fertilizer dataset.csv`

,metric,value
0,relative_path,data/raw/pre-combine/Crop and fertilizer datas...
1,size_mb,0.3594
2,rows,4513
3,columns,11
4,memory_mb,1.6806
5,exact_duplicate_rows,0


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,16
1,n,int64,0,0.0000,27
2,p,int64,0,0.0000,17
3,k,int64,0,0.0000,30
4,ph,float64,0,0.0000,7
5,temperature,int64,0,0.0000,7
6,rainfall,int64,0,0.0000,15
7,district_name,str,0,0.0000,5
8,soil_color,str,0,0.0000,7
9,fertilizer,str,0,0.0000,19


In [9]:
# File 2: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_02)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 0


In [10]:
# File 2: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_02)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,ph,temperature,rainfall,district_name,soil_color,fertilizer,link
0,sugarcane,75,50,100,6.5000,20,1000,Kolhapur,Black,Urea,https://youtu.be/2t5Am0xLTOo
1,sugarcane,80,50,100,6.5000,20,1000,Kolhapur,Black,Urea,https://youtu.be/2t5Am0xLTOo
2,sugarcane,85,50,100,6.5000,20,1000,Kolhapur,Black,Urea,https://youtu.be/2t5Am0xLTOo
3,sugarcane,90,50,100,6.5000,20,1000,Kolhapur,Black,Urea,https://youtu.be/2t5Am0xLTOo
4,sugarcane,95,50,100,6.5000,20,1000,Kolhapur,Black,Urea,https://youtu.be/2t5Am0xLTOo


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,"4,513.0000",95.4099,38.0606,20.0000,60.0000,105.0000,125.0000,150.0000
p,"4,513.0000",54.3419,16.5520,10.0000,40.0000,55.0000,65.0000,90.0000
k,"4,513.0000",63.5952,35.6919,5.0000,40.0000,55.0000,75.0000,150.0000
ph,"4,513.0000",6.7153,0.6252,5.5000,6.0000,6.5000,7.0000,8.5000
temperature,"4,513.0000",25.9151,5.8973,10.0000,20.0000,25.0000,30.0000,40.0000
rainfall,"4,513.0000",819.1890,251.7308,300.0000,600.0000,800.0000,"1,000.0000","1,700.0000"


...

,count,unique,top,freq
label,4513,16,sugarcane,1010
district_name,4513,5,Kolhapur,1430
soil_color,4513,7,Black,2260
fertilizer,4513,19,Urea,1364
link,4513,278,https://youtu.be/2t5Am0xLTOo,1010


**Top values: `label`**

,count
label,
sugarcane,1010
wheat,859
cotton,650
jowar,394
maize,350
rice,309
groundnut,177
tur,126
ginger,125


**Top values: `soil_color`**

,count
soil_color,
Black,2260
Red,744
Dark Brown,659
Red,480
Reddish Brown,265
Light Brown,54
Medium Brown,51


## File 3: `Crop_Recommendation.csv`

Folder: `.`  
Path: `data/raw/pre-combine/Crop_Recommendation.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [11]:
# File 3: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/Crop_Recommendation.csv'
df_03 = read_table(path)
display_structure(path, df_03)


### โครงสร้างไฟล์: `Crop_Recommendation.csv`

,metric,value
0,relative_path,data/raw/pre-combine/Crop_Recommendation.csv
1,size_mb,0.1360
2,rows,2200
3,columns,8
4,memory_mb,0.2522
5,exact_duplicate_rows,0


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,22
1,n,int64,0,0.0000,137
2,p,int64,0,0.0000,117
3,k,int64,0,0.0000,73
4,ph,float64,0,0.0000,2200
5,temperature,float64,0,0.0000,2200
6,rainfall,float64,0,0.0000,2200
7,humidity,float64,0,0.0000,2200


In [12]:
# File 3: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_03)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 0


In [13]:
# File 3: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_03)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,ph,temperature,rainfall,humidity
0,rice,90,42,43,6.5030,20.8797,202.9355,82.0027
1,rice,85,58,41,7.0381,21.7705,226.6555,80.3196
2,rice,60,55,44,7.8402,23.0045,263.9642,82.3208
3,rice,74,35,40,6.9804,26.4911,242.8640,80.1584
4,rice,78,42,42,7.6285,20.1302,262.7173,81.6049


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,"2,200.0000",50.5518,36.9173,0.0000,21.0000,37.0000,84.2500,140.0000
p,"2,200.0000",53.3627,32.9859,5.0000,28.0000,51.0000,68.0000,145.0000
k,"2,200.0000",48.1491,50.6479,5.0000,20.0000,32.0000,49.0000,205.0000
ph,"2,200.0000",6.4695,0.7739,3.5048,5.9717,6.4250,6.9236,9.9351
temperature,"2,200.0000",25.6162,5.0637,8.8257,22.7694,25.5987,28.5617,43.6755
rainfall,"2,200.0000",103.4637,54.9584,20.2113,64.5517,94.8676,124.2675,298.5601
humidity,"2,200.0000",71.4818,22.2638,14.2580,60.2620,80.4731,89.9488,99.9819


**Categorical summary**

,count,unique,top,freq
label,2200,22,rice,100


**Top values: `label`**

,count
label,
rice,100
maize,100
chickpea,100
kidneybeans,100
pigeonpeas,100
mothbeans,100
mungbean,100
blackgram,100
lentil,100


## File 5: `crop-yield.csv`

Folder: `.`  
Path: `data/raw/pre-combine/crop-yield.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [17]:
# File 5: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/crop-yield.csv'
df_05 = read_table(path)
display_structure(path, df_05)


### โครงสร้างไฟล์: `crop-yield.csv`

,metric,value
0,relative_path,data/raw/pre-combine/crop-yield.csv
1,size_mb,1.0384
2,rows,10000
3,columns,20
4,memory_mb,4.1124
5,exact_duplicate_rows,0


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,6
1,n,int64,0,0.0000,150
2,p,int64,0,0.0000,85
3,k,int64,0,0.0000,130
4,ph,float64,0,0.0000,341
5,temperature,float64,0,0.0000,2890
6,rainfall,float64,0,0.0000,9784
7,humidity,float64,0,0.0000,4874
8,soil_moisture,float64,0,0.0000,4302
9,soil_type,str,0,0.0000,4


In [18]:
# File 5: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_05)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 0


In [19]:
# File 5: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_05)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,ph,temperature,rainfall,humidity,soil_moisture,soil_type,organic_carbon,sunlight_hours,wind_speed,region,altitude,season,irrigation_type,fertilizer,pesticide,yield
0,maize,132,62,22,6.3500,22.9700,"1,305.6800",53.8900,59.7800,Clay,0.4300,7.7300,15.9600,Central,36,Rabi,Canal,223.4800,23.3600,11.4200
1,potato,122,71,66,5.9800,17.0000,"1,942.0500",76.9000,25.5400,Sandy,0.6500,9.2500,12.6000,North,1561,Rabi,Canal,161.5400,4.4200,23.1900
2,rice,44,35,104,8.0700,25.5200,"2,216.2000",44.7800,25.8700,Sandy,0.7900,8.5000,15.6300,North,1870,Rabi,Rainfed,184.6200,6.2900,7.9400
3,sugarcane,136,96,113,4.8300,18.5900,607.1800,31.8900,42.9700,Silt,0.4500,8.7500,5.4900,East,765,Kharif,Rainfed,274.0200,2.7200,72.5300
4,wheat,101,34,42,5.8400,22.7400,483.4700,46.2700,48.0100,Silt,0.6900,8.0000,7.4400,Central,1143,Zaid,Rainfed,72.6900,15.3700,6.7200


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,"10,000.0000",104.9830,43.2917,30.0000,68.0000,105.0000,142.0000,179.0000
p,"10,000.0000",56.8096,24.5230,15.0000,35.0000,57.0000,78.0000,99.0000
k,"10,000.0000",85.0087,37.5414,20.0000,52.0000,85.0000,118.0000,149.0000
ph,"10,000.0000",6.5043,0.9787,4.8000,5.6700,6.5200,7.3500,8.2000
temperature,"10,000.0000",25.0974,8.7190,10.0000,17.5900,24.9700,32.8300,40.0000
rainfall,"10,000.0000","1,546.6949",718.8660,300.7000,925.7675,"1,548.5500","2,165.4525","2,799.2600"
humidity,"10,000.0000",59.8942,17.4024,30.0100,44.8300,59.5500,75.0375,89.9900
soil_moisture,"10,000.0000",40.0837,14.4619,15.0000,27.6300,40.0000,52.7625,64.9900
organic_carbon,"10,000.0000",0.9584,0.3750,0.3000,0.6300,0.9600,1.2900,1.6000
sunlight_hours,"10,000.0000",7.5079,2.0065,4.0000,5.7700,7.5300,9.2200,11.0000


...

,count
label,
rice,1702
wheat,1671
potato,1670
cotton,1666
maize,1658
sugarcane,1633


**Top values: `season`**

,count
season,
Rabi,3355
Kharif,3350
Zaid,3295


**Top values: `soil_type`**

,count
soil_type,
Loamy,2543
Sandy,2496
Silt,2494
Clay,2467


## File 6: `crop_data.csv`

Folder: `.`  
Path: `data/raw/pre-combine/crop_data.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [20]:
# File 6: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/crop_data.csv'
df_06 = read_table(path)
display_structure(path, df_06)


### โครงสร้างไฟล์: `crop_data.csv`

,metric,value
0,relative_path,data/raw/pre-combine/crop_data.csv
1,size_mb,0.0237
2,rows,694
3,columns,7
4,memory_mb,0.0753
5,exact_duplicate_rows,48


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,97
1,n,int64,0,0.0000,67
2,p,int64,0,0.0000,54
3,k,int64,0,0.0000,70
4,ph,float64,0,0.0000,104
5,temperature,float64,0,0.0000,95
6,rainfall,float64,0,0.0000,137


In [21]:
# File 6: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_06)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 48


,label,n,p,k,ph,temperature,rainfall
13,aloe_vera,60,15,30,7.5000,18.0000,350.0000
14,aloe_vera,60,15,30,7.5000,18.0000,350.0000
21,ashwagandha,45,18,55,6.5000,20.0000,800.0000
23,ashwagandha,45,18,55,6.5000,20.0000,800.0000
36,bamboo,150,40,120,6.5000,25.0000,"2,000.0000"
38,bamboo,150,40,120,6.5000,25.0000,"2,000.0000"
71,castor,95,40,110,7.0000,20.0000,800.0000
73,castor,95,40,110,7.0000,20.0000,800.0000
111,giloy,80,30,70,6.8000,22.0000,850.0000
113,giloy,80,30,70,6.8000,22.0000,850.0000


In [22]:
# File 6: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_06)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,ph,temperature,rainfall
0,dragon_fruit,100,55,165,5.7000,32.0000,"1,050.0000"
1,dragon_fruit,100,60,160,6.5000,32.0000,900.0000
2,dragon_fruit,90,30,160,6.4000,31.0000,900.0000
3,dragon_fruit,80,40,150,6.0000,30.0000,800.0000
4,dragon_fruit,80,45,150,6.0000,34.0000,800.0000


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,694.0000,75.4870,39.6559,10.0000,50.0000,80.0000,100.0000,250.0000
p,694.0000,45.3127,22.6016,10.0000,30.0000,40.0000,60.0000,158.0000
k,694.0000,68.5000,61.4049,10.0000,30.0000,50.0000,92.0000,615.0000
ph,694.0000,6.0283,0.7299,3.8200,5.5200,6.1600,6.5400,7.6000
temperature,694.0000,25.4722,4.7045,8.0000,22.6760,26.0100,28.6400,38.0000
rainfall,694.0000,904.8645,546.2252,15.3400,579.7500,855.0000,"1,111.6800","2,817.8600"


**Categorical summary**

,count,unique,top,freq
label,694,97,papaya,15


**Top values: `label`**

,count
label,
papaya,15
sunflower,15
dragon_fruit,10
pomegranates,10
saffron_(kesar),10
sugarcane,10
brinjal,10
cabbage,10
cardamom,10


## File 7: `crop_yield.csv`

Folder: `.`  
Path: `data/raw/pre-combine/crop_yield.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [23]:
# File 7: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/crop_yield.csv'
df_07 = read_table(path)
display_structure(path, df_07)


### โครงสร้างไฟล์: `crop_yield.csv`

,metric,value
0,relative_path,data/raw/pre-combine/crop_yield.csv
1,size_mb,2.0617
2,rows,19689
3,columns,16
4,memory_mb,5.7131
5,exact_duplicate_rows,0


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,55
1,n,int64,0,0.0000,23
2,p,int64,0,0.0000,20
3,k,int64,0,0.0000,18
4,ph,float64,0,0.0000,24
5,temperature,float64,0,0.0000,460
6,rainfall,float64,0,0.0000,632
7,humidity,float64,0,0.0000,580
8,state,str,0,0.0000,30
9,year,int64,0,0.0000,24


In [24]:
# File 7: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_07)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 0


In [25]:
# File 7: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_07)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,ph,temperature,rainfall,humidity,state,year,season,area,production,fertilizer,pesticide,yield
0,arecanut,60,18,38,5.8000,22.4100,"1,468.9200",70.7100,Assam,1997,Whole Year,"73,814.0000",56708,"7,024,878.3800","22,882.3400",0.7961
1,arhar/tur,60,18,38,5.8000,22.4100,"1,468.9200",70.7100,Assam,1997,Kharif,"6,637.0000",4685,"631,643.2900","2,057.4700",0.7104
2,castor_seed,60,18,38,5.8000,22.4100,"1,468.9200",70.7100,Assam,1997,Kharif,796.0000,22,"75,755.3200",246.7600,0.2383
3,coconut,60,18,38,5.8000,22.4100,"1,468.9200",70.7100,Assam,1997,Whole Year,"19,656.0000",126905000,"1,870,661.5200","6,093.3600","5,238.0517"
4,cotton(lint),60,18,38,5.8000,22.4100,"1,468.9200",70.7100,Assam,1997,Kharif,"1,739.0000",794,"165,500.6300",539.0900,0.4209


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,"19,689.0000",76.6268,20.0606,50.0000,65.0000,72.0000,80.0000,150.0000
p,"19,689.0000",33.4993,11.5567,15.0000,24.0000,38.0000,43.0000,55.0000
k,"19,689.0000",32.2345,7.9155,20.0000,25.0000,33.0000,38.0000,50.0000
ph,"19,689.0000",6.6403,0.6906,5.5000,6.0000,6.6000,7.1000,8.0000
temperature,"19,689.0000",24.0080,4.4080,6.2600,22.9100,25.5600,26.8100,28.6300
rainfall,"19,689.0000","1,337.9994",637.5852,249.2400,894.1600,"1,172.0200","1,613.4000","5,244.3600"
humidity,"19,689.0000",64.6760,11.4741,34.4700,54.6200,68.1100,73.7300,86.0600
year,"19,689.0000","2,009.1276",6.4981,"1,997.0000","2,004.0000","2,010.0000","2,015.0000","2,020.0000"
area,"19,689.0000","179,926.5703","732,828.6759",0.5000,"1,390.0000","9,317.0000","75,112.0000","50,808,100.0000"
production,"19,689.0000","16,435,941.2731","263,056,839.8126",0.0000,"1,393.0000","13,804.0000","122,718.0000","6,326,000,000.0000"


...

,count
label,
rice,1197
maize,975
moong(green_gram),740
urad,733
groundnut,725
sesamum,685
potato,628
sugarcane,605
wheat,545


**Top values: `state`**

,count
state,
Karnataka,1432
Andhra Pradesh,1266
West Bengal,1094
Chhattisgarh,915
Bihar,896
Madhya Pradesh,845
Uttar Pradesh,825
Tamil Nadu,822
Gujarat,817


**Top values: `season`**

,count
season,
Kharif,8232
Rabi,5742
Whole Year,3717
Summer,1195
Autumn,414
Winter,389


## File 8: `data_core.csv`

Folder: `.`  
Path: `data/raw/pre-combine/data_core.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [26]:
# File 8: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/data_core.csv'
df_08 = read_table(path)
display_structure(path, df_08)


### โครงสร้างไฟล์: `data_core.csv`

,metric,value
0,relative_path,data/raw/pre-combine/data_core.csv
1,size_mb,0.3497
2,rows,8000
3,columns,9
4,memory_mb,1.8050
5,exact_duplicate_rows,0


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,11
1,n,int64,0,0.0000,46
2,p,int64,0,0.0000,47
3,k,int64,0,0.0000,24
4,temperature,float64,0,0.0000,1816
5,humidity,float64,0,0.0000,3004
6,soil_moisture,float64,0,0.0000,3723
7,soil_type,str,0,0.0000,5
8,fertilizer,str,0,0.0000,7


In [27]:
# File 8: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_08)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 0


In [28]:
# File 8: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_08)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,temperature,humidity,soil_moisture,soil_type,fertilizer
0,maize,37,0,0,26.0000,52.0000,38.0000,Sandy,Urea
1,sugarcane,12,36,0,29.0000,52.0000,45.0000,Loamy,DAP
2,cotton,7,30,9,34.0000,65.0000,62.0000,Black,14-35-14
3,tobacco,22,20,0,32.0000,62.0000,34.0000,Red,28-28
4,paddy,35,0,0,28.0000,54.0000,46.0000,Clayey,Urea


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,"8,000.0000",18.4291,11.8524,0.0000,9.0000,14.0000,26.0000,46.0000
p,"8,000.0000",18.5125,13.2441,0.0000,8.0000,18.0000,30.0000,46.0000
k,"8,000.0000",3.9164,5.4948,0.0000,0.0000,1.0000,5.0000,23.0000
temperature,"8,000.0000",30.3389,4.4783,20.0000,27.0500,30.2400,33.4600,40.0000
humidity,"8,000.0000",59.2107,8.1774,40.0200,53.2775,59.1100,65.0825,80.0000
soil_moisture,"8,000.0000",43.5809,12.5962,20.0000,33.9675,42.2500,52.9500,70.0000


...

,count,unique,top,freq
label,8000,11,sugarcane,763
soil_type,8000,5,Clayey,1623
fertilizer,8000,7,14-35-14,1188


**Top values: `label`**

,count
label,
sugarcane,763
maize,753
wheat,747
ground_nuts,732
pulses,728
cotton,722
millets,718
tobacco,717
oil_seeds,711


**Top values: `soil_type`**

,count
soil_type,
Clayey,1623
Black,1613
Red,1594
Loamy,1590
Sandy,1580


## File 9: `original_dataset.csv`

Folder: `.`  
Path: `data/raw/pre-combine/original_dataset.csv`

สำรวจไฟล์นี้แบบเบื้องต้น โดยแบ่งเป็น 3 ส่วนย่อย: โครงสร้างไฟล์, missing/duplicate และตัวอย่างกับสถิติพื้นฐาน


In [29]:
# File 9: load และดูโครงสร้างไฟล์
path = PROJECT_ROOT / 'data/raw/pre-combine/original_dataset.csv'
df_09 = read_table(path)
display_structure(path, df_09)


### โครงสร้างไฟล์: `original_dataset.csv`

,metric,value
0,relative_path,data/raw/pre-combine/original_dataset.csv
1,size_mb,0.2263
2,rows,5000
3,columns,9
4,memory_mb,0.8752
5,exact_duplicate_rows,4000


**Columns / dtypes / missing / unique**

,column,dtype,missing_count,missing_pct,unique_count
0,label,str,0,0.0000,10
1,n,int64,0,0.0000,121
2,p,int64,0,0.0000,72
3,k,int64,0,0.0000,53
4,ph,float64,0,0.0000,262
5,temperature,float64,0,0.0000,736
6,rainfall,float64,0,0.0000,938
7,humidity,float64,0,0.0000,871
8,soil_type,str,0,0.0000,4


In [30]:
# File 9: ตรวจ missing values และ exact duplicate rows
display_missing_duplicates(df_09)


### Missing values และ duplicate rows

ไม่พบ missing values
Exact duplicate rows: 4000


,label,n,p,k,ph,temperature,rainfall,humidity,soil_type
0,guinea_corn,40,27,45,5.8900,21.6600,112.4300,94.7900,sandy loam
1,pepper,120,7,47,6.6500,24.2500,54.7700,83.0400,loamy clay
2,tomatoes,102,11,47,6.5000,27.9900,27.1500,92.7800,sandy loam
4,yam,36,43,21,7.1400,28.3600,52.9300,84.8600,loamy
5,maize,94,54,17,5.8700,23.3900,107.3200,61.7400,sandy loam
6,beans,17,59,17,5.6900,18.4200,132.9800,23.4300,clay
7,maize,63,35,16,6.2700,22.0300,83.7300,65.3600,sandy loam
8,yam,10,56,16,6.8300,28.1700,36.3600,81.0500,loamy
9,tomatoes,106,20,51,6.3400,29.7300,20.4900,90.9700,sandy loam
10,guinea_corn,10,5,42,6.8900,20.2400,109.2500,91.0900,sandy loam


In [31]:
# File 9: ดูตัวอย่างข้อมูลและสถิติพื้นฐาน
display_samples_and_stats(df_09)


### ตัวอย่างข้อมูลและสถิติพื้นฐาน

**Head sample**

,label,n,p,k,ph,temperature,rainfall,humidity,soil_type
0,guinea_corn,40,27,45,5.8900,21.6600,112.4300,94.7900,sandy loam
1,pepper,120,7,47,6.6500,24.2500,54.7700,83.0400,loamy clay
2,tomatoes,102,11,47,6.5000,27.9900,27.1500,92.7800,sandy loam
3,orange,5,25,6,6.0100,30.7200,106.8100,94.0100,loamy
4,yam,36,43,21,7.1400,28.3600,52.9300,84.8600,loamy


**Numeric summary**

,count,mean,std,min,25%,50%,75%,max
n,"5,000.0000",51.2264,34.9694,0.0000,22.0000,39.0000,84.0000,120.0000
p,"5,000.0000",40.8416,22.7551,5.0000,20.0000,41.0000,60.0000,80.0000
k,"5,000.0000",36.2420,21.1747,5.0000,19.0000,36.0000,49.0000,85.0000
ph,"5,000.0000",6.5597,0.6502,5.0100,6.1000,6.5000,6.9500,8.8700
temperature,"5,000.0000",23.7725,4.5632,10.0100,19.8675,24.3650,27.4450,34.9500
rainfall,"5,000.0000",91.3347,55.2718,20.2100,56.0200,80.1100,108.4800,298.5600
humidity,"5,000.0000",68.3773,28.6770,14.2600,59.2500,83.1200,90.0200,95.0000


...

,count,unique,top,freq
label,5000,10,soybean,587
soil_type,5000,4,sandy loam,1836


**Top values: `label`**

,count
label,
soybean,587
beans,584
guinea_corn,561
pepper,538
orange,507
tomatoes,491
yam,486
maize,485
rice,462


**Top values: `soil_type`**

,count
soil_type,
sandy loam,1836
loamy,1580
clay,1046
loamy clay,538


## 3. Notes สำหรับขั้นตอนถัดไป

หลังจากรัน notebook นี้แล้ว ค่อยเลือกว่าจะ combine ไฟล์ไหนเข้าด้วยกัน โดยดูจากชื่อ column ที่ซ้ำกัน/ต่างกัน, target ที่ต้องใช้ เช่น `label`, และระดับความสะอาดของแต่ละไฟล์ เช่น missing values หรือ duplicate rows
